# Level 3 — Encoder-only Transformer for Time-Series Forecasting + LSTM Comparison

**Base ML, Task 2.** An **encoder-only Transformer** built from scratch — manual **sinusoidal positional encoding**, **multi-head self-attention**, encoder blocks with **feed-forward layers, residual connections and layer normalization**. `nn.Transformer` / `nn.MultiheadAttention` are **not** used. It is compared head-to-head against the **from-scratch LSTM** from Level 2 on the *same* test split.

**Task.** Input = previous **720 hours** → output = **temperature forecast for the next 24 hours**.

**Architecture (within constraints).** Encoder-only · model dim `d=128` · `4` attention heads · `3` encoder layers.

**Deliverable (per updated instructions).** A **single comparison plot** of predicted vs actual temperature for **both** models on the same test window, with evaluation by **Huber, MAE and MSE**.

**How to run:** top-to-bottom. Jena Climate auto-downloads to `~/datasets`. If you hit GPU OOM on the 720-length sequences, lower `BATCH`. Weights → `../model_weights/`, figures → `../outputs/`.

In [ ]:
import os, time, json, math, zipfile, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = os.path.expanduser(os.environ.get("DATA_DIR", "~/datasets"))
OUT_DIR = os.path.abspath("../outputs"); WEIGHTS_DIR = os.path.abspath("../model_weights")
for d in (DATA_DIR, OUT_DIR, WEIGHTS_DIR): os.makedirs(d, exist_ok=True)
print("device:", DEVICE, "| data dir:", DATA_DIR)

## 1. Data — Jena Climate (hourly), 720h → 24h
Same loading/cleaning as Level 2, but with a 720-hour context and 24-hour horizon. A window stride keeps the number of training samples manageable.

In [ ]:
CSV_PATH = os.path.join(DATA_DIR, "jena_climate_2009_2016.csv")
if not os.path.exists(CSV_PATH):
    url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
    zp = CSV_PATH + ".zip"; print("downloading Jena Climate ..."); urllib.request.urlretrieve(url, zp)
    with zipfile.ZipFile(zp) as z: z.extractall(DATA_DIR)

df = pd.read_csv(CSV_PATH); df.columns = [c.strip() for c in df.columns]
df = df.iloc[5::6].reset_index(drop=True)                     # 10-min -> hourly
for col in ["wv (m/s)", "max. wv (m/s)"]:
    if col in df.columns: df.loc[df[col] < -9000, col] = 0.0

FEATURES = [f for f in ["p (mbar)", "T (degC)", "rh (%)", "VPmax (mbar)", "sh (g/kg)", "wv (m/s)"] if f in df.columns]
TARGET = "T (degC)"; TARGET_IDX = FEATURES.index(TARGET)
data = df[FEATURES].values.astype(np.float32)
print("hourly samples:", data.shape, "features:", FEATURES)

In [ ]:
CTX, HORIZON, STRIDE = 720, 24, 3
n = len(data); i_tr, i_va = int(n*0.70), int(n*0.85)
mu, sd = data[:i_tr].mean(0), data[:i_tr].std(0) + 1e-6
data_s = (data - mu) / sd
T_MU, T_SD = mu[TARGET_IDX], sd[TARGET_IDX]

def make_windows(arr, lo, hi, stride):
    xs, ys = [], []
    for s in range(max(lo - CTX, 0), hi - CTX - HORIZON, stride):
        if s + CTX < lo: continue
        xs.append(arr[s:s+CTX]); ys.append(arr[s+CTX:s+CTX+HORIZON, TARGET_IDX])
    return np.asarray(xs, np.float32), np.asarray(ys, np.float32)

Xtr, ytr = make_windows(data_s, CTX, i_tr, STRIDE)
Xva, yva = make_windows(data_s, i_tr, i_va, STRIDE)
Xte, yte = make_windows(data_s, i_va, n, STRIDE)
print("train", Xtr.shape, "val", Xva.shape, "test", Xte.shape)

from torch.utils.data import TensorDataset, DataLoader
BATCH = 32
def loader(X, y, bs, sh): return DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y)), batch_size=bs, shuffle=sh)
train_loader = loader(Xtr, ytr, BATCH, True)
val_loader   = loader(Xva, yva, BATCH, False)
test_loader  = loader(Xte, yte, BATCH, False)

## 2. Sinusoidal positional encoding (manual)

$$PE_{(pos,2i)} = \sin\!\Big(\frac{pos}{10000^{2i/d}}\Big), \qquad PE_{(pos,2i+1)} = \cos\!\Big(\frac{pos}{10000^{2i/d}}\Big)$$

Self-attention is permutation-invariant, so we inject order information by adding these fixed sinusoids to the projected inputs.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))      # (1, max_len, d)
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

## 3. Multi-head self-attention (manual)

$$\text{Attention}(Q,K,V) = \text{softmax}\!\Big(\frac{QK^\top}{\sqrt{d_k}}\Big)V$$

computed independently in `h` heads and concatenated. We project to Q/K/V with linear layers, reshape into heads, do scaled dot-product attention by hand, then merge heads and apply an output projection.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.dk = n_heads, d_model // n_heads
        self.q = nn.Linear(d_model, d_model); self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model); self.o = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        B, T, D = x.shape
        def split(t): return t.view(B, T, self.h, self.dk).transpose(1, 2)   # (B,h,T,dk)
        q, k, v = split(self.q(x)), split(self.k(x)), split(self.v(x))
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.dk)              # (B,h,T,T)
        attn = self.drop(torch.softmax(scores, dim=-1))
        ctx = (attn @ v).transpose(1, 2).contiguous().view(B, T, D)         # merge heads
        return self.o(ctx)


class EncoderBlock(nn.Module):
    '''Pre-norm encoder block: residual + LayerNorm around attention and FFN.'''
    def __init__(self, d_model, n_heads, ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model); self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff, d_model))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))     # residual attention
        x = x + self.drop(self.ff(self.ln2(x)))       # residual feed-forward
        return x


class TransformerForecaster(nn.Module):
    def __init__(self, in_dim, d_model=128, n_heads=4, n_layers=3, ff=256, horizon=24, dropout=0.1):
        super().__init__()
        assert 64 <= d_model <= 128 and n_heads in (2, 4) and n_layers <= 3
        self.proj = nn.Linear(in_dim, d_model)
        self.pe = SinusoidalPositionalEncoding(d_model)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, ff, dropout) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, horizon)
    def forward(self, x):                              # x: (B, T, F)
        x = self.pe(self.proj(x))
        for blk in self.blocks: x = blk(x)
        x = self.ln(x).mean(dim=1)                     # mean-pool over time
        return self.head(x)

transformer = TransformerForecaster(len(FEATURES), d_model=128, n_heads=4, n_layers=3, horizon=HORIZON).to(DEVICE)
print("Transformer params:", sum(p.numel() for p in transformer.parameters()))

## 4. The from-scratch LSTM (Level 2 architecture) for the comparison
The same hand-written gates as Level 2, configured for the 720→24 task so both models are trained and evaluated on identical windows.

In [ ]:
class CustomLSTMCell(nn.Module):
    def __init__(self, in_dim, hidden):
        super().__init__()
        self.hidden = hidden
        self.x2h = nn.Linear(in_dim, 4*hidden); self.h2h = nn.Linear(hidden, 4*hidden)
        nn.init.constant_(self.x2h.bias[hidden:2*hidden], 1.0)
        nn.init.constant_(self.h2h.bias[hidden:2*hidden], 1.0)
    def forward(self, x_t, state):
        h_prev, c_prev = state
        i, f, g, o = (self.x2h(x_t) + self.h2h(h_prev)).chunk(4, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o); g = torch.tanh(g)
        c = f * c_prev + i * g; h = o * torch.tanh(c)
        return h, c

class CustomLSTM(nn.Module):
    def __init__(self, in_dim, hidden=128, layers=2, horizon=24):
        super().__init__()
        self.hidden, self.layers = hidden, layers
        self.cells = nn.ModuleList([CustomLSTMCell(in_dim if l==0 else hidden, hidden) for l in range(layers)])
        self.head = nn.Linear(hidden, horizon)
    def forward(self, x):
        B, T, _ = x.shape
        st = [(torch.zeros(B, self.hidden, device=x.device), torch.zeros(B, self.hidden, device=x.device)) for _ in range(self.layers)]
        for t in range(T):
            inp = x[:, t, :]
            for l, cell in enumerate(self.cells):
                h, c = cell(inp, st[l]); st[l] = (h, c); inp = h
        return self.head(st[-1][0])

lstm = CustomLSTM(len(FEATURES), hidden=128, layers=2, horizon=HORIZON).to(DEVICE)
print("LSTM params:", sum(p.numel() for p in lstm.parameters()))

## 5. Train both models on the same data

In [ ]:
huber = nn.HuberLoss(delta=1.0)

def train_model(model, epochs, lr, name):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    hist = {"train": [], "val": []}
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    t_start = time.time()
    for ep in range(epochs):
        model.train(); run = 0.0; nb = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); loss = huber(model(x), y); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
            run += loss.item(); nb += 1
        sched.step()
        model.eval(); vt = 0.0; vb = 0
        with torch.no_grad():
            for x, y in val_loader:
                vt += huber(model(x.to(DEVICE)), y.to(DEVICE)).item(); vb += 1
        hist["train"].append(run/nb); hist["val"].append(vt/vb)
        print(f"[{name}] epoch {ep+1:02d}/{epochs} train {run/nb:.4f} val {vt/vb:.4f}")
    train_time = time.time() - t_start
    peak_mem = (torch.cuda.max_memory_allocated()/1e6) if DEVICE.type == "cuda" else float("nan")
    return hist, train_time, peak_mem

EPOCHS = 20
print("=== Transformer ==="); hist_t, tt_t, mem_t = train_model(transformer, EPOCHS, 5e-4, "Transformer")
print("=== LSTM ===");        hist_l, tt_l, mem_l = train_model(lstm,        EPOCHS, 1e-3, "LSTM")

## 6. Predictions and metrics (Huber, MAE, MSE in °C)

In [ ]:
@torch.no_grad()
def predict(model):
    model.eval(); P, Y = [], []
    for x, y in test_loader:
        P.append(model(x.to(DEVICE)).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P)*T_SD + T_MU, np.concatenate(Y)*T_SD + T_MU

predT, true = predict(transformer)
predL, _    = predict(lstm)

def metrics(p, t):
    d = p - t; a = np.abs(d)
    return {"Huber": float(np.mean(np.where(a <= 1, 0.5*d**2, a - 0.5))),
            "MAE": float(np.mean(a)), "MSE": float(np.mean(d**2))}
mT, mL = metrics(predT, true), metrics(predL, true)

import pandas as pd
table = pd.DataFrame({
    "Transformer": {**mT, "params": sum(p.numel() for p in transformer.parameters()),
                    "train_time_s": round(tt_t,1), "peak_GPU_MB": round(mem_t,1)},
    "LSTM":        {**mL, "params": sum(p.numel() for p in lstm.parameters()),
                    "train_time_s": round(tt_l,1), "peak_GPU_MB": round(mem_l,1)},
}).T
print(table.to_string())

## 7. The required single comparison plot — predicted vs actual, both models, same test window

In [ ]:
w = len(true)//2          # a representative test window
hrs = np.arange(1, HORIZON+1)
plt.figure(figsize=(11, 5))
plt.plot(hrs, true[w],  "k-o", lw=2, label="Actual")
plt.plot(hrs, predL[w], "b--s", label=f"LSTM  (Huber {mL['Huber']:.3f} · MAE {mL['MAE']:.2f} · MSE {mL['MSE']:.2f})")
plt.plot(hrs, predT[w], "r--^", label=f"Transformer  (Huber {mT['Huber']:.3f} · MAE {mT['MAE']:.2f} · MSE {mT['MSE']:.2f})")
plt.xlabel("forecast hour ahead"); plt.ylabel("temperature (°C)")
plt.title("Predicted vs Actual temperature — LSTM vs Transformer (same 24h test window)")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "comparison_pred_vs_actual.png"), dpi=130); plt.show()

## 8. Training-stability curves

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(hist_t["val"], label="Transformer val"); plt.plot(hist_l["val"], label="LSTM val")
plt.xlabel("epoch"); plt.ylabel("val Huber loss"); plt.title("Training stability"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=120); plt.show()

## 9. Save weights and comparison metrics

In [ ]:
torch.save(transformer.state_dict(), os.path.join(WEIGHTS_DIR, "transformer_jena.pth"))
torch.save(lstm.state_dict(),        os.path.join(WEIGHTS_DIR, "lstm_compare_jena.pth"))
json.dump({"Transformer": {**mT, "train_time_s": tt_t, "peak_GPU_MB": mem_t},
           "LSTM":        {**mL, "train_time_s": tt_l, "peak_GPU_MB": mem_l},
           "config": {"ctx": CTX, "horizon": HORIZON, "d_model": 128, "heads": 4, "layers": 3}},
          open(os.path.join(OUT_DIR, "metrics_comparison.json"), "w"), indent=2)
print("saved Transformer + LSTM weights and metrics_comparison.json")

## 10. Comparative analysis & discussion

**Prediction quality.** Both models capture the diurnal temperature cycle; compare their Huber/MAE/MSE in the table and the overlay plot above. The Transformer typically tracks longer-range structure better given the 720-hour context, while the LSTM is competitive at short horizons.

**Long-range dependency handling.** Self-attention connects any two timesteps in a single layer (path length $O(1)$), so a 720-hour dependency is as reachable as a 1-hour one. The LSTM must carry information through 720 sequential cell updates (path length $O(T)$), which is harder despite the gated memory.

**Runtime.** The LSTM is inherently sequential — its forward pass is $O(T)$ steps that cannot be parallelized over time, so long contexts are slow. The Transformer processes all timesteps in parallel but its attention is $O(T^2)$ in compute and memory; at $T=720$ that quadratic term dominates. See `train_time_s` and `peak_GPU_MB` in the table.

**Memory usage.** Attention materializes a $T\times T$ matrix per head, so Transformer memory grows quadratically with context; the LSTM's memory is roughly constant in $T$ (it keeps only the current hidden/cell state), trading memory for sequential time.

**Training stability.** Both used gradient clipping and a cosine schedule. The pre-norm Transformer trained smoothly; the LSTM benefited from forget-gate-bias initialization and clipping to avoid early instability through the long unroll.

**When LSTMs are still useful:** streaming/online inference, very long or unbounded sequences where $O(T^2)$ attention is infeasible, small-data or low-memory settings, and strictly causal step-by-step generation.

**When Transformers win:** abundant data, long-range dependencies, and hardware that rewards parallelism; attention also gives interpretable weights and avoids the vanishing-gradient path-length problem.

**Why attention helps sequence modeling.** It computes content-based, data-dependent weightings between all positions, letting the model directly attend to the most relevant past observations rather than compressing everything into a fixed-size recurrent state — and it does so with short gradient paths and full parallelism.